# Article-grouped XGBoost rankers

Fit one joint curator/audience LambdaMART model for root candidates and one for all comments. The frozen split and shared transformed predictors are created by 06A2_shared_model_preprocessing.ipynb and consumed unchanged by 06B and 06C. Within development, a reproducible broad random search is followed by a local grid; both stages use the same five article folds and resumable checkpoints. Query groups are article × selector.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pyarrow.parquet as pq

from commentgap_analysis.ranking import run_ranker_workflow

SEED = int(os.getenv("COMMENTGAP_MODEL_SEED", "20260813"))
FEATURE_ROOT = Path(os.getenv("COMMENTGAP_FEATURE_ROOT", "model_output/selection_2025/features"))
MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_XGB_ROOT", "model_output/selection_2025/xgboost_paper2"))
DEVICE = os.getenv("COMMENTGAP_XGB_DEVICE", "auto")
BOOTSTRAP_DRAWS = int(os.getenv("COMMENTGAP_BOOTSTRAP_DRAWS", "1000"))
BROAD_SEARCH_CONFIGS = int(os.getenv("COMMENTGAP_XGB_BROAD_CONFIGS", "32"))
REFINEMENT_TOP_CONFIGS = int(os.getenv("COMMENTGAP_XGB_REFINEMENT_TOP", "5"))
FORCE_RECOMPUTE = os.getenv("COMMENTGAP_XGB_FORCE_RECOMPUTE", "0").lower() in {"1", "true", "yes"}
feature_manifest = json.loads((MODEL_DATA_ROOT / "feature_manifest.json").read_text())
preprocessing = json.loads((MODEL_DATA_ROOT / "preprocessing_parameters.json").read_text())
assert preprocessing["version"] == 4, "Force-rerun 06A2 shared preprocessing"
provenance = json.loads((FEATURE_ROOT / "provenance_manifest.json").read_text())
aqua_expected_features = sorted(
    feature
    for feature in feature_manifest["features"]
    if feature.startswith("aqua_")
    and feature.endswith("_expected")
    and feature != "aqua_score_expected"
)
assert len(aqua_expected_features) == 20, aqua_expected_features
provenance["watermark"]

'INFERENCE'

## Validation design

Articles common to the root and all-comment scopes are split exactly 50/50. Hard strata are article month × a joint root/all candidate-size tercile. Seeded assignment is accepted only when absolute standardized mean differences are at most 0.05 for log root size, log all-comment size, both scope-specific pick counts, and reply proportion. The development half receives shared five-fold assignments for tuning; one frozen development model is then applied once to the sealed Paper 2 half.

In [2]:
article_split = pq.read_table(MODEL_DATA_ROOT / "master_article_split.parquet").to_pandas()
import pandas as pd
split_balance = pd.read_csv(MODEL_DATA_ROOT / "split_balance_diagnostics.csv")
fold_feature_columns = {int(fold): mapping for fold, mapping in preprocessing["fold_feature_columns"].items()}
assert set(fold_feature_columns) == set(range(5)), fold_feature_columns

results = {}
for scope in ("root", "all"):
    choice_path = MODEL_DATA_ROOT / f"choice_set_{scope}.parquet"
    choice_set = pq.read_table(choice_path).to_pandas()
    features = list(feature_manifest["models"][scope]["features"])
    active_fold_columns = {
        fold: {feature: source for feature, source in mapping.items() if feature in features}
        for fold, mapping in fold_feature_columns.items()
    }
    required_columns = set(features) | {source for mapping in active_fold_columns.values() for source in mapping.values()}
    missing_columns = sorted(required_columns - set(choice_set.columns))
    assert not missing_columns, f"{scope} choice set is missing model columns: {missing_columns}"
    required_author_features = {
        "log_author_prior_30d_comments",
        "author_prior_30d_upvote_reception",
        "author_prior_30d_downvote_reception",
        "log_author_prior_comments_story",
    }
    missing_author = sorted(required_author_features - set(features))
    assert not missing_author, f"{scope} author-history contract is incomplete: {missing_author}"
    retired_author_features = {
        "log_author_prior_30d_snapshot_upvotes",
        "log_author_prior_30d_snapshot_downvotes",
    }
    assert not retired_author_features.intersection(features)
    required_activity_features = {"discussion_pace"}
    assert not required_activity_features - set(features)
    retired_activity_features = {
        "log_prior_comments", "log_comments_prev_hour", "recent_activity_share",
        "log_branch_prior_comments", "log_branch_comments_prev_hour",
    }
    assert not retired_activity_features.intersection(features)
    missing_aqua = sorted(set(aqua_expected_features) - set(features))
    assert not missing_aqua, (
        f"{scope} primary model omits AQuA expected dimensions; "
        "re-run commentgap-features with the production AQuA store: "
        f"{missing_aqua}"
    )
    assert "aqua_score_expected" not in features
    results[scope] = run_ranker_workflow(
        choice_set,
        features,
        OUTPUT_ROOT,
        scope=scope,
        article_split=article_split,
        device=DEVICE,
        seed=SEED,
        bootstrap_draws=BOOTSTRAP_DRAWS,
        broad_search_count=BROAD_SEARCH_CONFIGS,
        refinement_top_configs=REFINEMENT_TOP_CONFIGS,
        force_recompute=FORCE_RECOMPUTE,
        fold_feature_columns=active_fold_columns,
    )
    del choice_set
{
    "articles_by_role": article_split["split_role"].value_counts().to_dict(),
    "max_abs_smd": split_balance["abs_standardized_mean_difference"].max(),
    "models": results,
}

broad_random: configuration 1/32 (5 development folds)
broad_random: configuration 2/32 (5 development folds)
broad_random: configuration 3/32 (5 development folds)
broad_random: configuration 4/32 (5 development folds)
broad_random: configuration 5/32 (5 development folds)
broad_random: configuration 6/32 (5 development folds)
broad_random: configuration 7/32 (5 development folds)
broad_random: configuration 8/32 (5 development folds)
broad_random: configuration 9/32 (5 development folds)
broad_random: configuration 10/32 (5 development folds)
broad_random: configuration 11/32 (5 development folds)
broad_random: configuration 12/32 (5 development folds)
broad_random: configuration 13/32 (5 development folds)
broad_random: configuration 14/32 (5 development folds)
broad_random: configuration 15/32 (5 development folds)
broad_random: configuration 16/32 (5 development folds)
broad_random: configuration 17/32 (5 development folds)
broad_random: configuration 18/32 (5 development folds)
b

{'articles_by_role': {'development': 2559, 'paper2_test': 2559},
 'max_abs_smd': np.float64(0.0461283770203233),
 'models': {'root': {'scope': 'root',
   'seed': 20260813,
   'device': 'cuda',
   'features': ['log_words',
    'sentiment_positive',
    'sentiment_negative',
    'toxicity_probability',
    'lexdiv_length_adjusted',
    'reading_level_length_adjusted',
    'url_present',
    'article_similarity_top3',
    'novelty_prior_roots_model',
    'log_hours_since_article',
    'prior_reply_composition',
    'discussion_pace',
    'vienna_overnight',
    'vienna_weekday_shoulder_evening',
    'vienna_weekend_day_evening',
    'log_author_prior_30d_comments',
    'author_prior_30d_upvote_reception',
    'author_prior_30d_downvote_reception',
    'log_author_prior_comments_story',
    'aqua_relevance_expected',
    'aqua_fact_expected',
    'aqua_opinion_expected',
    'aqua_justification_expected',
    'aqua_solution_proposal_expected',
    'aqua_additional_knowledge_expected',
    

## Artifact checks

The master split must be shared exactly by both scopes. Each scope must contain one development-only model plus sealed-test scores, article metrics, bootstrap summaries, grouped permutation importance, TreeSHAP samples, and tie-sensitivity metrics. No all-data model is created in this confirmatory workflow. Fingerprinted stage metadata in workflow_cache.json reuses matching final models and test outputs; set COMMENTGAP_XGB_FORCE_RECOMPUTE=1 only when deliberate recomputation is required.

In [3]:
role_counts = article_split["split_role"].value_counts().to_dict()
assert set(role_counts) == {"development", "paper2_test"}
assert abs(role_counts["development"] - role_counts["paper2_test"]) <= 1
assert split_balance["accepted"].all()
for scope in ("root", "all"):
    scope_root = OUTPUT_ROOT / scope
    required = [
        scope_root / "development_model.json",
        scope_root / "workflow_cache.json",
        scope_root / "development_cv_broad_random.csv",
        scope_root / "development_broad_configurations.json",
        scope_root / "development_narrow_configurations.json",
        scope_root / "development_cv_narrow_grid.csv",
        scope_root / "development_cv_summary.csv",
        scope_root / "test_scores_wide.parquet",
        scope_root / "test_metric_summary.parquet",
        scope_root / "test_grouped_permutation_importance.parquet",
        scope_root / "test_treeshap_sample.parquet",
        scope_root / "test_tie_sensitivity_metrics.parquet",
        scope_root / "model_manifest.json",
    ]
    assert all(path.exists() for path in required), [
        str(path) for path in required if not path.exists()
    ]
    manifest = json.loads((scope_root / "model_manifest.json").read_text())
    assert manifest["reported_scores"] == "sealed_paper2_test"
    assert manifest["hyperparameter_search"]["strategy"] == "broad_random_then_narrow_grid"
    assert manifest["hyperparameter_search"]["test_used_for_selection"] is False
    assert set(manifest["cache"]["stage_status"].values()) <= {"computed", "reused"}
    assert manifest["development_articles"] == role_counts["development"]
    assert manifest["paper2_test_articles"] == role_counts["paper2_test"]
{scope: json.loads((OUTPUT_ROOT / scope / "model_manifest.json").read_text()) for scope in ("root", "all")}

{'root': {'best_parameters': {'colsample_bytree': 0.564266,
   'gamma': 0.0,
   'learning_rate': 0.043909,
   'max_depth': 5,
   'min_child_weight': 45.14844,
   'n_estimators': 385,
   'reg_alpha': 0.0,
   'reg_lambda': 3.18456,
   'subsample': 0.964784},
  'cache': {'development_fingerprint': '55fcfb1a9b91cc485f1782b3fae6d6fd85677a0352707d09cabcd8718b0a18b4',
   'force_recompute': False,
   'metadata_artifact': 'workflow_cache.json',
   'stage_status': {'development_model': 'reused',
    'test_grouped_permutation': 'reused',
    'test_metrics': 'reused',
    'test_predictions': 'reused',
    'test_tie_sensitivity': 'reused',
    'test_treeshap': 'reused'},
   'test_feature_fingerprint': '2801a36cb7b8f4e8f667e504b702e2669de712949b71153204b90aae4a5a2e82',
   'test_label_fingerprint': '764525df38e640ddcfc1203fd3966144234be3e7326e70053c5cdaba6f627746',
   'tie_label_fingerprint': '91f73d04cd44f98cf154b1f70cd1842836924b7af14f41f5d3eb6d2f249284eb'},
  'choice_set_fingerprint': '0e8efcfc60b

## Next step

After the Rmd regressions and both rankers finish, run 06D_model_tables_plots.ipynb. Only test_scores_wide.parquet and the corresponding test metrics are eligible for Paper 2 predictive-performance and FORUM claims. The saved development_model.json has never trained on a Paper 2 test article.